In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
import math
import igraph as ig

In [ ]:
plt.rcParams["mathtext.fontset"] = "cm"
plt.rc('font', family='serif',size=10)
plt.rc('text', usetex=True)

Code for finding average path length for BA model

In [ ]:
N_vals = np.array([i for i in range(5, 50)])
avg_path_lengths = np.zeros_like(N_vals, dtype=np.float64)

for k in range(100):
    for i in range(5, 50):
        M = 2*i*(i+1)
        start = ig.Graph(directed=True)
        start.add_vertices(i)

        not_okay = True

        while(not_okay):
            ba_graph = ig.Graph.Barabasi(M, 2, directed=True, start_from=start)
            in_degrees = ba_graph.indegree()
            zero_in = [j for j, d in enumerate(in_degrees) if d == 0]
            out_degrees = ba_graph.outdegree()
            zero_out = [j for j, d in enumerate(out_degrees) if d == 0]

            if(len(set(zero_in).intersection(set(zero_out))) == 0):
                not_okay = False

        end_nodes = [j for j in range(i)]
        start_nodes = [ba_graph.vcount() - i + j for j in range(i)]

        avg_path = 0
        for v1 in start_nodes:
            min_path = 500000
            for v2 in end_nodes:
                path_length = len(ba_graph.get_shortest_path(v1, v2))
                if (path_length > 0 and path_length < min_path):
                    min_path = path_length
            
            avg_path += min_path
        
        avg_path = avg_path / i
        avg_path_lengths[i-5] += avg_path

avg_path_lengths = avg_path_lengths / 100

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)

ax.plot(N_vals, avg_path_lengths, color='black')
#ax.scatter(N_vals, connect_vals, color='black')

ax.set_ylabel("Average Distance", fontsize=15)
ax.set_xlabel("Number of Start Nodes", fontsize=15)
ax.set_title("Distance Between Start and End Nodes", fontsize=15)

ax.set_xlim(0, 50)
#ax.legend()
fig.tight_layout()

Different start node choice below: consider every pair of start nodes and average over.

In [ ]:
M = 50
not_okay = True

while(not_okay):
    ba_graph = ig.Graph.Barabasi(M, 1, directed=True)
    ba_copy = ba_graph.as_undirected()
    degrees = ba_copy.degree()
    degree_nums = [d for j, d in enumerate(degrees)]

    if(degree_nums[0] == max(degree_nums)):
        not_okay = False

p_vals = np.arange(0, 1.001, 0.05)

total_op_commute = np.zeros_like(p_vals)
total_eq_commute = np.zeros_like(p_vals)
R = 10

in_degrees = ba_graph.indegree()
start_nodes = [j for j, d in enumerate(in_degrees) if d == 0]
ineq_matrix, ineq_vector = bound_condition(ba_graph)

for i in range(R):
    print(i/R * 100)

    eq_commute_times = np.zeros_like(p_vals)
    op_commute_times = np.zeros_like(p_vals)

    for m in range(len(p_vals)):

        road_types = assign_road_type(ba_graph, p_vals[m])
        quadratic_coefficients = quad_coeff(road_types)
        linear_coefficients = linear_coeff(road_types)

        for l in range(len(start_nodes)):
            for k in range(l+1, len(start_nodes)):
                start_pair = [start_nodes[l], start_nodes[k]]

                eq_matrix, eq_vector = kirchoff_law_condition(ba_graph, start_pair, [0])

                x = cp.Variable(ba_graph.ecount())

                P = quadratic_coefficients
                q = linear_coefficients
                G = ineq_matrix
                h = ineq_vector
                A = eq_matrix
                b = eq_vector
                eq_prob = cp.Problem(cp.Minimize((1/2)*cp.quad_form(x, P) + q.T @ x),
                                [G @ x <= h,
                                A @ x == b])
                eq_prob.solve()

                sol = np.array(x.value)
                rounded_sol = np.where(sol < 0, 0, sol)

                eq_commute = commute_time(rounded_sol, road_types)

                y = cp.Variable(ba_graph.ecount())
                P = 2*P
                op_prob = cp.Problem(cp.Minimize((1/2)*cp.quad_form(y, P) + q.T @ y),
                                [G @ y <= h,
                                A @ y == b])
                op_prob.solve()

                sol = np.array(y.value)
                rounded_sol = np.where(sol < 0, 0, sol)

                op_commute = commute_time(rounded_sol, road_types)

                eq_commute_times[m] += eq_commute
                op_commute_times[m] += op_commute

        eq_commute_times[m] = eq_commute_times[m] / math.comb(len(start_nodes), 2)
        op_commute_times[m] = op_commute_times[m] / math.comb(len(start_nodes), 2)

    total_eq_commute += eq_commute_times
    total_op_commute += op_commute_times


avg_eq_commute = total_eq_commute / R
avg_op_commute = total_op_commute / R